## Build the Training Dataset

### Let's load the processed dataset

In [1]:
import pandas as pd
import numpy as np

played_matches = pd.read_csv("../data/processed/played_matches.csv")
played_matches["date"] = pd.to_datetime(played_matches["date"])

played_matches.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,target
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,0,1
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,0,2
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,0,2
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,0,1
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,0,2


### I'll make sure it is sorted by date

In [2]:
played_matches = played_matches.sort_values("date")
played_matches = played_matches.reset_index(drop=True)

### Create a team history table

#### Instead of constantly checking: "home team, away team", I'll make one table where every row represents one team's performance in one match.
#### Then convert every match into two rows


In [4]:
team_history = []

for _, row in played_matches.iterrows():
    
    # Home team
    team_history.append({
        "date": row["date"],
        "team": row["home_team"],
        "opponent": row["away_team"],
        "goals_for": row["home_score"],
        "goals_against": row["away_score"],
        "is_home": 1,
        "neutral": row["neutral"]
    })

    # Away team
    team_history.append({
        "date": row["date"],
        "team": row["away_team"],
        "opponent": row["home_team"],
        "goals_for": row["away_score"],
        "goals_against": row["home_score"],
        "is_home": 0,
        "neutral": row["neutral"]
    })

team_history = pd.DataFrame(team_history)

team_history.head()

,date,team,opponent,goals_for,goals_against,is_home,neutral
0,1872-11-30,Scotland,England,0.0,0.0,1,0
1,1872-11-30,England,Scotland,0.0,0.0,0,0
2,1873-03-08,England,Scotland,4.0,2.0,1,0
3,1873-03-08,Scotland,England,2.0,4.0,0,0
4,1874-03-07,Scotland,England,2.0,1.0,1,0


In [5]:
team_history["win"] = (
    team_history["goals_for"] >
    team_history["goals_against"]
).astype(int)

team_history["draw"] = (
    team_history["goals_for"] ==
    team_history["goals_against"]
).astype(int)

team_history["loss"] = (
    team_history["goals_for"] <
    team_history["goals_against"]
).astype(int)

#### Average goals in the last 10 matches

In [7]:
team_history["avg_goals_last10"] = (
    team_history
    .groupby("team")["goals_for"]
    .transform(
        lambda x:
        x.shift(1)  # This's one of the most important lines in the project. It prevents the data leakage.
         .rolling(10, min_periods=1)
         .mean()
    )
)

#### Goals conceded

In [8]:
team_history["avg_goals_against_last10"] = (
    team_history
    .groupby("team")["goals_against"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(10, min_periods=1)
         .mean()
    )
)

#### Win rate

In [9]:
team_history["win_rate_last10"] = (
    team_history
    .groupby("team")["win"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(10, min_periods=1)
         .mean()
    )
)

#### Goal difference

In [10]:
team_history["goal_difference"] = (
    team_history["goals_for"] -
    team_history["goals_against"]
)

team_history["avg_goal_difference_last10"] = (
    team_history
    .groupby("team")["goal_difference"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(10, min_periods=1)
         .mean()
    )
)

In [11]:
team_history.head(15)

,date,team,opponent,goals_for,goals_against,is_home,neutral,win,draw,loss,avg_goals_last10,avg_goals_against_last10,win_rate_last10,goal_difference,avg_goal_difference_last10
0,1872-11-30,Scotland,England,0.0,0.0,1,0,0,1,0,NaN,NaN,NaN,0.0,NaN
1,1872-11-30,England,Scotland,0.0,0.0,0,0,0,1,0,NaN,NaN,NaN,0.0,NaN
2,1873-03-08,England,Scotland,4.0,2.0,1,0,1,0,0,0.000000,0.000000,0.000000,2.0,0.000000
3,1873-03-08,Scotland,England,2.0,4.0,0,0,0,0,1,0.000000,0.000000,0.000000,-2.0,0.000000
4,1874-03-07,Scotland,England,2.0,1.0,1,0,1,0,0,1.000000,2.000000,0.000000,1.0,-1.000000
5,1874-03-07,England,Scotland,1.0,2.0,0,0,0,0,1,2.000000,1.000000,0.500000,-1.0,1.000000
6,1875-03-06,England,Scotland,2.0,2.0,1,0,0,1,0,1.666667,1.333333,0.333333,0.0,0.333333
7,1875-03-06,Scotland,England,2.0,2.0,0,0,0,1,0,1.333333,1.666667,0.333333,0.0,-0.333333
8,1876-03-04,Scotland,England,3.0,0.0,1,0,1,0,0,1.500000,1.750000,0.250000,3.0,-0.250000
9,1876-03-04,England,Scotland,0.0,3.0,0,0,0,0,1,1.750000,1.500000,0.250000,-3.0,0.250000


In [12]:
team_history.columns

Index(['date', 'team', 'opponent', 'goals_for', 'goals_against', 'is_home',
       'neutral', 'win', 'draw', 'loss', 'avg_goals_last10',
       'avg_goals_against_last10', 'win_rate_last10', 'goal_difference',
       'avg_goal_difference_last10'],
      dtype='object')

In [13]:
team_history = team_history.fillna(0)

In [14]:
team_history.isnull().sum()

date                          0
team                          0
opponent                      0
goals_for                     0
goals_against                 0
is_home                       0
neutral                       0
win                           0
draw                          0
loss                          0
avg_goals_last10              0
avg_goals_against_last10      0
win_rate_last10               0
goal_difference               0
avg_goal_difference_last10    0
dtype: int64

In [15]:
# After removing NA
team_history.head(15)

,date,team,opponent,goals_for,goals_against,is_home,neutral,win,draw,loss,avg_goals_last10,avg_goals_against_last10,win_rate_last10,goal_difference,avg_goal_difference_last10
0,1872-11-30,Scotland,England,0.0,0.0,1,0,0,1,0,0.000000,0.000000,0.000000,0.0,0.000000
1,1872-11-30,England,Scotland,0.0,0.0,0,0,0,1,0,0.000000,0.000000,0.000000,0.0,0.000000
2,1873-03-08,England,Scotland,4.0,2.0,1,0,1,0,0,0.000000,0.000000,0.000000,2.0,0.000000
3,1873-03-08,Scotland,England,2.0,4.0,0,0,0,0,1,0.000000,0.000000,0.000000,-2.0,0.000000
4,1874-03-07,Scotland,England,2.0,1.0,1,0,1,0,0,1.000000,2.000000,0.000000,1.0,-1.000000
5,1874-03-07,England,Scotland,1.0,2.0,0,0,0,0,1,2.000000,1.000000,0.500000,-1.0,1.000000
6,1875-03-06,England,Scotland,2.0,2.0,1,0,0,1,0,1.666667,1.333333,0.333333,0.0,0.333333
7,1875-03-06,Scotland,England,2.0,2.0,0,0,0,1,0,1.333333,1.666667,0.333333,0.0,-0.333333
8,1876-03-04,Scotland,England,3.0,0.0,1,0,1,0,0,1.500000,1.750000,0.250000,3.0,-0.250000
9,1876-03-04,England,Scotland,0.0,3.0,0,0,0,0,1,1.750000,1.500000,0.250000,-3.0,0.250000


#### Right now we have two rows per match:
#### Brazil
#### Spain
#### I'll convert them back into one row per match, but with the engineered features.

#### First I'll Create two DataFrames: Home team features, Away team features.

In [16]:
home_features = team_history[team_history["is_home"] == 1].copy()

In [17]:
away_features = team_history[team_history["is_home"] == 0].copy()

#### Then will rename the columns.

In [18]:
home_features = home_features.rename(columns={
    "team": "home_team",
    "avg_goals_last10": "home_avg_goals",
    "avg_goals_against_last10": "home_avg_goals_against",
    "win_rate_last10": "home_win_rate",
    "avg_goal_difference_last10": "home_goal_difference"
})

In [19]:
away_features = away_features.rename(columns={
    "team": "away_team",
    "avg_goals_last10": "away_avg_goals",
    "avg_goals_against_last10": "away_avg_goals_against",
    "win_rate_last10": "away_win_rate",
    "avg_goal_difference_last10": "away_goal_difference"
})

### Training data
#### Now I'll merge them with the original matches.

In [20]:
training_data = played_matches.merge(
    home_features[
        [
            "date",
            "home_team",
            "home_avg_goals",
            "home_avg_goals_against",
            "home_win_rate",
            "home_goal_difference"
        ]
    ],
    on=["date", "home_team"]
)

In [21]:
training_data = training_data.merge(
    away_features[
        [
            "date",
            "away_team",
            "away_avg_goals",
            "away_avg_goals_against",
            "away_win_rate",
            "away_goal_difference"
        ]
    ],
    on=["date", "away_team"]
)

#### I'll only keep the columns we'll train on initially.

In [22]:
training_data = training_data[
    [
        "home_avg_goals",
        "away_avg_goals",
        "home_avg_goals_against",
        "away_avg_goals_against",
        "home_win_rate",
        "away_win_rate",
        "home_goal_difference",
        "away_goal_difference",
        "neutral",
        "target"
    ]
]

#### Let's inspect it

In [23]:
training_data.head()

,home_avg_goals,away_avg_goals,home_avg_goals_against,away_avg_goals_against,home_win_rate,away_win_rate,home_goal_difference,away_goal_difference,neutral,target
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,1
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,2
2,1.000000,2.000000,2.000000,1.000000,0.000000,0.500000,-1.000000,1.000000,0,2
3,1.666667,1.333333,1.333333,1.666667,0.333333,0.333333,0.333333,-0.333333,0,1
4,1.500000,1.750000,1.750000,1.500000,0.250000,0.250000,-0.250000,0.250000,0,2


#### Let's add career matches played. This solves the issue we discussed earlier: distinguishing between a team with no history and a team with a poor record.

In [49]:
team_history["career_matches"] = (
    team_history
    .groupby("team")
    .cumcount()
)

In [50]:
team_history.head(10)

,date,team,opponent,goals_for,goals_against,is_home,neutral,win,draw,loss,avg_goals_last10,avg_goals_against_last10,win_rate_last10,goal_difference,avg_goal_difference_last10,career_matches
0,1872-11-30,Scotland,England,0.0,0.0,1,0,0,1,0,0.000000,0.000000,0.000000,0.0,0.000000,0
1,1872-11-30,England,Scotland,0.0,0.0,0,0,0,1,0,0.000000,0.000000,0.000000,0.0,0.000000,0
2,1873-03-08,England,Scotland,4.0,2.0,1,0,1,0,0,0.000000,0.000000,0.000000,2.0,0.000000,1
3,1873-03-08,Scotland,England,2.0,4.0,0,0,0,0,1,0.000000,0.000000,0.000000,-2.0,0.000000,1
4,1874-03-07,Scotland,England,2.0,1.0,1,0,1,0,0,1.000000,2.000000,0.000000,1.0,-1.000000,2
5,1874-03-07,England,Scotland,1.0,2.0,0,0,0,0,1,2.000000,1.000000,0.500000,-1.0,1.000000,2
6,1875-03-06,England,Scotland,2.0,2.0,1,0,0,1,0,1.666667,1.333333,0.333333,0.0,0.333333,3
7,1875-03-06,Scotland,England,2.0,2.0,0,0,0,1,0,1.333333,1.666667,0.333333,0.0,-0.333333,3
8,1876-03-04,Scotland,England,3.0,0.0,1,0,1,0,0,1.500000,1.750000,0.250000,3.0,-0.250000,4
9,1876-03-04,England,Scotland,0.0,3.0,0,0,0,0,1,1.750000,1.500000,0.250000,-3.0,0.250000,4


#### Add it to the training dataset

In [51]:
home_features = team_history[team_history["is_home"] == 1].copy()
away_features = team_history[team_history["is_home"] == 0].copy()

In [52]:
home_features = home_features.rename(columns={
    "team": "home_team",
    "avg_goals_last10": "home_avg_goals",
    "avg_goals_against_last10": "home_avg_goals_against",
    "win_rate_last10": "home_win_rate",
    "avg_goal_difference_last10": "home_goal_difference",
    "career_matches": "home_career_matches"
})

away_features = away_features.rename(columns={
    "team": "away_team",
    "avg_goals_last10": "away_avg_goals",
    "avg_goals_against_last10": "away_avg_goals_against",
    "win_rate_last10": "away_win_rate",
    "avg_goal_difference_last10": "away_goal_difference",
    "career_matches": "away_career_matches"
})

In [53]:
home_features.columns

Index(['date', 'home_team', 'opponent', 'goals_for', 'goals_against',
       'is_home', 'neutral', 'win', 'draw', 'loss', 'home_avg_goals',
       'home_avg_goals_against', 'home_win_rate', 'goal_difference',
       'home_goal_difference', 'home_career_matches'],
      dtype='object')

In [54]:
training_data = played_matches.merge(
    home_features[
        [
            "date",
            "home_team",
            "home_avg_goals",
            "home_avg_goals_against",
            "home_win_rate",
            "home_goal_difference",
            "home_career_matches"      # <-- The new column
        ]
    ],
    on=["date", "home_team"]
)

In [55]:
training_data = training_data.merge(
    away_features[
        [
            "date",
            "away_team",
            "away_avg_goals",
            "away_avg_goals_against",
            "away_win_rate",
            "away_goal_difference",
            "away_career_matches"      # <-- The new column
        ]
    ],
    on=["date", "away_team"]
)

In [56]:
training_data = training_data[
    [
        "home_avg_goals",
        "away_avg_goals",
        "home_avg_goals_against",
        "away_avg_goals_against",
        "home_win_rate",
        "away_win_rate",
        "home_goal_difference",
        "away_goal_difference",
        "home_career_matches",     # <-- The first new column
        "away_career_matches",     # <-- The second new column
        "neutral",
        "target"
    ]
]

### Saving the training dataset

In [57]:
training_data.to_csv(
    "../data/processed/training_data.csv",
    index=False
)